In [15]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}

div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:40px;}
</style>
"""))

# 5. 생성형 AI 평가 : 
- 첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
- 두번째 체인 :  음식 -> 음식의 레시피
- 최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피

In [13]:
def recipe():
    country = input("나라입력하세요 : ")
    '국가명을 입력받아 그 국가의 유명한 음식 레시피 return'
    from langchain_ollama import ChatOllama
    from langchain_core.prompts import PromptTemplate
    from langchain_core.output_parsers import StrOutputParser  
    from langchain_core.output_parsers import JsonOutputParser
    llm= ChatOllama(model="exaone3.5:2.4b")
    
    #첫번째 체인 :  나라이름 -> 그 나라에서 가장 유명한 음식 
    food_prompt_template = PromptTemplate(
        template="{국가}에서 가장 유명한 음식이 무엇입니까? 출력은 음식이름 하나만! 출력해줘.",
        input_variables =['국가'])
    outputParser=StrOutputParser()
    famous_food_chain= food_prompt_template | llm | outputParser
    #두번째 체인 :  음식 -> 음식의 레시피
    recipe_prompt_template = PromptTemplate(
                    template="""다음 음식({음식이름})에 대한 정보를 제공해줘,
                            1. 음식 이름
                            2. 재료 준비 (Prep)
                            3. 조리 (Cook)
                            4. 마무리 (Finish) 
                추가 텍스트 없이 오직 유효한 JSON 객체만 반환해. 
                예시 형식:
                {{
                      "음식이름": "{음식이름}",
                      "1.재료 준비 (Prep)": "채소를 얇게 채 썬다.",
                      "2.조리 (Cook)": "채소와 소고기를 각각 볶은 뒤 계란 프라이를 만든다.",
                      "3.마무리 (Finish) ": "따뜻한 밥 위에 재료들과 계란 프라이, 고추장, 참기름을 올린다."
                }}""",
                    input_variables =['음식이름']
                    )
    recipe_output_parser=JsonOutputParser()
    recipe_chain= recipe_prompt_template | llm | recipe_output_parser
    #최종 체인 : 나라이름 -> 그 나라에 가장 유명한 음식의 레시피
    final_chain = famous_food_chain | recipe_chain
    return final_chain.invoke({'국가': country})

In [14]:
recipe()

나라입력하세요 : 태국


{'음식이름': '카오피아오',
 '1.재료 준비 (Prep)': '밥 준비, 소고기 얇게 썰기, 배추나 무를 얇게 채 썰기, 계란, 고추장, 참기름',
 '2.조리 (Cook)': '소고기를 팬에 볶은 후, 채소와 함께 볶다가 계란을 풀어 팬에 부어 익혀내고, 고추장과 참기름으로 양념한다',
 '3.마무리 (Finish)': '따뜻한 밥 위에 볶은 소고기와 채소를 얹고, 계란이 익으면서 생긴 후커드 소스와 참기름을 뿌려 완성'}